# Chapter 11 — Tensor Index Transformations

Part 4: Broadcasting and tensor manipulation.

**Ladder:** explicit index maps and host views (Method 01) → Triton tile-copy, row-gather, and scatter-add kernels (Method 02) → fused transform-plus-compute kernels with explicit strides (Method 03).

**Thread.**

- **Recurring idea.** A transform changes the index map `addr(i) = o + Σ i_k s_k`, not the values.
- **Recurring decision.** Is the result a view (same buffer, new shape/stride/offset) or a copy (new contiguous buffer)?
- **Recurring question.** What are the new shape, stride, and storage offset?
- **Chain.** flatten/unflatten → reshape → squeeze/expand → transpose/permute → slice/reverse → gather/scatter/index-select → non-contiguous strides → fused kernels.
- **Payoff.** Every Triton kernel here is the same load/store skeleton with a different address formula.

**Study tip.** Fix one $3\times4$ matrix for the whole chapter:

$$
X=\begin{bmatrix}0&1&2&3\\4&5&6&7\\8&9&10&11\end{bmatrix},
\qquad \operatorname{stride}(X)=(4,1).
$$

- Write the flatten map by hand: $f(i,j)=iN+j=i\cdot4+j$.
- Write the transpose map by hand: $X^\top_{ij}=X_{ji}=4j+i$, with shape $(4,3)$ and stride $(1,4)$.
- If both lines are right, every other transform is a variation on them.

**Learning objectives**

- Read a shape change as a triple $(\text{shape},\text{stride},\text{offset})$ and decide whether it is a view or a copy.
- Derive flatten/unflatten, squeeze/unsqueeze, transpose/permute, and slice/reverse index maps.
- State when shape and stride are preserved, when a stride is swapped, and when a stride becomes zero.
- Write gather, scatter, scatter-add, and index selection as explicit address formulas.
- Write a tiled Triton transpose, a row-gather kernel, and an atomic scatter-add kernel.
- Fuse a transform with arithmetic so the data is read and written once.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch10_broadcasting/10_complete.ipynb) · [Next →](../../part5_layouts/ch12_layouts/12_complete.ipynb)

**Estimated time:** 70–85 min · **Difficulty:** Intermediate

**Prerequisites**

- Chapter 6: pointers, offsets, strides, masks, `tl.load`, and `tl.store`.
- Chapter 10: broadcasting, size-1 axes, and zero-stride views.
- Comfort with row-major indexing: $f=iN+j$.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Chapter map

**Thread.** Read the `Formula` and `Shape/stride rule` columns together; the `Kernel/demo` column names the cell that checks the claim.

| § | Transform | Formula | Shape/stride rule | Kernel/demo |
|---|---|---|---|---|
| **11.1** | Flatten | $f=iN+j$ | $(M,N)\to(MN,)$; stride $(1,)$ when contiguous | verification A |
| **11.2** | Unflatten | $i=f\mathbin{//}N,\ j=f\bmod N$ | $(MN,)\to(M,N)$; stride $(N,1)$ | verification A |
| **11.3** | Reshape | same flat index $f$ | $\prod n_k=\prod m_k$; contiguous target has stride $(m_1,1)$ | verification A |
| **11.4** | Squeeze | drop $n_p=1$ | remaining strides unchanged | verification B |
| **11.5** | Expand dims | insert $1$ at $p$ | new stride arbitrary; $0$ after `expand` | verification B |
| **11.6** | Transpose | $Y_{ij}=X_{ji}$ | $(M,N)\to(N,M)$; $(s_0,s_1)\to(s_1,s_0)$ | transpose tile kernel |
| **11.7** | Permute | $Y_{j_0,\dots}=X_{i_0,\dots}$ | $\operatorname{shape}_k=\operatorname{shape}_{\pi_k}$; same for strides | verification B |
| **11.8** | Slice | $Y_{ij}=X_{a+ip,\,b+jq}$ | $o'=o+as_0+bs_1$; $(ps_0,qs_1)$ | verification C |
| **11.9** | Reverse | $Y_{ij}=X_{M-1-i,N-1-j}$ | negative stride view, or a flip copy | verification C |
| **11.10** | Gather | $Y_{ij}=X_{\mathrm{idx}[i,j],j}$ | output shape = index shape | row-gather kernel |
| **11.11** | Scatter | $\operatorname{dst}[\mathrm{idx}[i,j],j]=\operatorname{src}[i,j]$ | destination indexed by data | atomic scatter-add kernel |
| **11.12** | Index selection | $Y_{k,:}=X_{\mathrm{idx}[k],:}$ | one whole slice per index | row-gather kernel |
| **11.13** | Non-contiguous | $\operatorname{addr}=o+\sum_k i_k s_k$ | pass the real strides | strided ReLU kernel |
| **★** | Worked example | all of the above | one $3\times4$ and one $2\times3\times4$ spine | asserts only |

## Concept diagrams

**Thread.** Logical shape = the legal indices; storage = a flat buffer; strides = the bridge.

- **Logical shape vs storage order.** The logical tensor is a set of indices; the storage is a flat buffer.

```text
logical view X (3 x 4)                  storage (12 floats, row-major)
      j=0  j=1  j=2  j=3
i=0  [ 0    1    2    3 ]               offset: 0  1  2  3  4  5  6  7  8  9 10 11
i=1  [ 4    5    6    7 ]               value : 0  1  2  3  4  5  6  7  8  9 10 11
i=2  [ 8    9   10   11 ]
                                        addr(i,j) = 4i + j
stride = (4, 1)                         offset = 0
```

- **Flatten and unflatten.** One map goes down to a flat index; the other recovers the pair.

```text
flatten:    (i, j) --f = i*4 + j-->  f          (1,2) --> 6
unflatten:  f --i = f//4, j = f%4--> (i, j)     10   --> (2,2)
```

- **Transpose and permute.** The values do not move; the index map and the strides change.

```text
transpose                              permute (2,0,1) on (2,3,4)
Y[i,j] = X[j,i]                        P[j,b,i] = A[b,i,j]
shape  (3,4)   -> (4,3)                shape  (2,3,4) -> (4,2,3)
stride (4,1)   -> (1,4)                stride (12,4,1) -> (1,12,4)
```

- **Gather and scatter.** A gather reads one row per index; a scatter writes one row per index; a scatter-add accumulates when indices repeat.

```text
gather:       out[i,j]     = src[idx[i,j], j]     read
scatter:      dst[idx[i,j], j] = src[i,j]         write (unique indices)
scatter-add:  dst[idx[i,j], j] += src[i,j]        accumulate (duplicates)
```

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import pytest
import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, check, require_gpu, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)

## Part I — Mathematics / Concepts

**Thread.**

- **Object.** A tensor is its shape $(n_0,\dots,n_{d-1})$, strides $(s_0,\dots,s_{d-1})$, and storage offset $o$.
- **Address.** $\operatorname{addr}(i_0,\dots,i_{d-1})=o+\sum_{k=0}^{d-1} i_k s_k$.
- **Contiguity.** Row-major contiguous means $s_{d-1}=1$, $s_k=n_{k+1}s_{k+1}$ for $0\le k\lt d-1$, and $o=0$.
- **View or copy.** A view keeps the buffer and changes $(\operatorname{shape},\operatorname{stride},o)$; a copy allocates a new contiguous buffer.
- **Central rule.** A transform is a change of the index map, not a change of the numbers.

The chapter spine is the $3\times4$ matrix

$$
X=\begin{bmatrix}0&1&2&3\\4&5&6&7\\8&9&10&11\end{bmatrix},
\qquad \operatorname{stride}(X)=(4,1),\qquad o=0.
$$

**Section template.** Every subsection uses the same bullets: Connection → Rule → Formula → Example → Where → Triton → Failure → Bridge. The numerical cells between the groups verify every shape and stride with `assert`, not with printed dumps.

### 11.1 Flatten

- **Connection.** Builds on Chapter 6's flat offset `i*N+j`: flatten names that map, and §11.2 will invert it.
- **Rule.** Reinterpret an $(M,N)$ matrix as a vector of length $MN$ in row-major order.
- **Formula.** $f(i,j)=iN+j$ for $0\le i\lt M$ and $0\le j\lt N$, with $\operatorname{flat}(X)_{f(i,j)}=X_{ij}$.
- **Example.** On the spine, $f(1,2)=1\cdot4+2=6$ and $f(2,3)=11$, so $\operatorname{flat}(X)=(0,1,2,3,4,5,6,7,8,9,10,11)$.
- **Where.** Turns a 2-D tile into the 1-D index space a kernel loops over: the first step of a dense layer, a loss, or a flat `tl.load`.
- **Triton.** For a contiguous matrix the flat address is just `offs`, so `tl.load(x_ptr + offs)` needs no multiply; for a view, compute `addr = r*s0 + c*s1`.
- **Failure.** `.view(-1)` raises on a non-contiguous tensor, while `.reshape(-1)` silently copies. The flat order is the *logical* row-major order, not the storage order: `X.t().reshape(-1)` is $(0,4,8,1,5,9,\dots)$.
- **Bridge.** §11.2 recovers the pair $(i,j)$ from $f$.

### 11.2 Unflatten

- **Connection.** Inverts §11.1 Flatten.
- **Rule.** Recover the coordinates of a flat index $f$.
- **Formula.** $i=f\mathbin{//}N$, $j=f\bmod N$, and $f=iN+j$.
- **Example.** $f=7\Rightarrow(1,3)$; $f=10\Rightarrow(2,2)$; $f=4\Rightarrow(1,0)$; check $2\cdot4+2=10$.
- **Where.** A kernel that owns a contiguous block of flat indices needs the row and column to apply a per-row transform, a mask, or a stride.
- **Triton.** `row = offs // N`, `col = offs % N`; pass `N` as a `tl.constexpr` when possible, and use `>>` and `&` for a power of two.
- **Failure.** Dividing by the number of rows instead of the row length; Triton integer `//` truncates toward zero, so normalize negative indices first.
- **Bridge.** §11.3 keeps $f$ fixed and changes how it is split into extents.

### 11.3 Reshape

- **Connection.** Builds on flatten/unflatten: a reshape keeps the flat index $f$ and changes the shape that splits it.
- **Rule.** Change the shape while preserving the row-major flat order; the total number of elements must match.
- **Formula.** For shapes $\mathbf n=(n_0,\dots,n_{d-1})$ and $\mathbf m=(m_0,\dots,m_{e-1})$: $\prod_k n_k=\prod_k m_k$, $f=\sum_k i_k\prod_{l\gt k}n_l=\sum_k j_k\prod_{l\gt k}m_l$, and $\operatorname{addr}_{\text{new}}(\mathbf j)=\operatorname{addr}_{\text{old}}(\mathbf i)$ for the same $f$.
- **Example.**

| target | values | stride |
|---|---|---|
| $(2,6)$ | `[[0,1,2,3,4,5],[6,7,8,9,10,11]]` | $(6,1)$ |
| $(4,3)$ | `[[0,1,2],[3,4,5],[6,7,8],[9,10,11]]` | $(3,1)$ |

  The $(4,3)$ result is **not** $X^\top$.
- **Where.** Attention reshapes $(B,S,H,D)\to(B,S,HD)$ and back; a linear layer reshapes a batch to a matrix.
- **Triton.** A reshape changes the host-side shape and the strides you compute, not the kernel body; for a contiguous target the address is the usual `r*N + c`.
- **Failure.** Using `reshape` when you mean `transpose` follows the flat order, not the transposed order; a reshape of a non-contiguous view may copy, costing a full read and write.
- **Bridge.** §11.4 removes a size-1 axis, the simplest reshape.

### 11.4 Squeeze

- **Connection.** Builds on §11.3 Reshape: removing a size-1 axis is a reshape with no data movement.
- **Rule.** Remove one or all axes of size 1.
- **Formula.** If $n_p=1$, the shape drops axis $p$ and the stride drops $s_p$: $\operatorname{shape}(Y)=(n_0,\dots,\widehat{n_p},\dots,n_{d-1})$ and $\operatorname{stride}(Y)=(s_0,\dots,\widehat{s_p},\dots,s_{d-1})$. The address is unchanged because $i_p s_p=0\cdot s_p=0$.
- **Example.** $S$ of shape $(2,1,3)$ has strides $(3,3,1)$; `squeeze(1)` gives shape $(2,3)$ and strides $(3,1)$, and $S_{i,0,j}=3i+0\cdot3+j=3i+j$.
- **Where.** Drop a singleton batch or channel axis before a 2-D kernel; remove the head axis after single-head attention.
- **Triton.** Squeeze on the host; a kernel sees only `M`, `N`, and the strides, and a size-1 axis contributes nothing to the address.
- **Failure.** Squeezing an axis that is not 1 is an error or a no-op depending on the call; squeezing the wrong size-1 axis gives a plausible shape with the wrong layout.
- **Bridge.** §11.5 is the inverse operation.

### 11.5 Expand dims

- **Connection.** Inverts §11.4 Squeeze: insert a size-1 axis.
- **Rule.** Insert a new axis of size 1 at position $p$.
- **Formula.** $\operatorname{shape}(Y)=(n_0,\dots,n_{p-1},1,n_p,\dots)$ and $\operatorname{stride}(Y)=(s_0,\dots,s_{p-1},\sigma,s_p,\dots)$, where $\sigma$ is arbitrary for a size-1 axis; after `expand` to size $m\gt 1$ on that axis, the stride becomes $0$.
- **Example.** For $v=(10,20,30)$ with stride $(1,)$: $v_{[:,None]}$ has shape $(3,1)$ and stride $(1,1)$; $v_{[None,:]}$ has shape $(1,3)$ and stride $(3,1)$; and `v[None,:].expand(2,3)` has shape $(2,3)$ and stride $(0,1)$.
- **Where.** Turns a row vector into a column (Chapter 10), adds a batch axis, and broadcasts a bias.
- **Triton.** `v[:, None]` and `v[None, :]` are offset-construction tools; Triton broadcasts size-1 axes, and a zero-stride axis needs no address term.
- **Failure.** Swapping `[:, None]` and `[None, :]` gives a valid shape with the wrong pairing, so the values are silently permuted.
- **Bridge.** §11.6 swaps axes instead of adding one.

In [ ]:
X = torch.arange(12).view(3, 4)
flat = X.reshape(-1)

assert X.shape == (3, 4) and X.stride() == (4, 1)
assert flat.tolist() == list(range(12))
assert X.reshape(2, 6).tolist() == [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11]]
assert X.reshape(2, 6).stride() == (6, 1)
assert X.reshape(6, 2).shape == (6, 2)
assert X.view(4, 3).tolist() == [[0, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]]
assert X.view(4, 3).tolist() != X.t().tolist()

Y = X.t()
assert Y.reshape(-1).tolist() == [0, 4, 8, 1, 5, 9, 2, 6, 10, 3, 7, 11]
try:
    Y.view(-1)
except RuntimeError:
    pass
else:
    raise AssertionError("view(-1) must reject a non-contiguous tensor")

rows = flat // 4
cols = flat % 4
assert rows.tolist() == [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]
assert cols.tolist() == [0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3]
assert (rows * 4 + cols).tolist() == flat.tolist()

print("shapes/strides: flatten, unflatten, reshape match the text")

### 11.6 Transpose

- **Connection.** Builds on the stride arithmetic of §11.3–§11.5; transpose swaps the last two axes.
- **Rule.** Swap the last two axes.
- **Formula.** $Y_{ij}=X_{ji}$, $\operatorname{shape}(Y)=(N,M)$, $\operatorname{stride}(Y)=(s_1,s_0)$, $o_Y=o_X$, and $\operatorname{addr}_Y(i,j)=o+i s_1+j s_0=\operatorname{addr}_X(j,i)$.
- **Example.**

$$
X^\top=\begin{bmatrix}0&4&8\\1&5&9\\2&6&10\\3&7&11\end{bmatrix},
\qquad \operatorname{stride}=(1,4).
$$

  $Y_{2,1}=X_{1,2}=6$, and both addresses equal $2\cdot1+1\cdot4=1\cdot4+2=6$.
- **Where.** $QK^\top$ in attention, covariance matrices, and switching between row- and column-major GEMM layouts.
- **Triton.** `tl.trans(tile)` transposes a register tile, and the global store decides whether the write is coalesced; alternatively, keep the tile and swap the strides in the store address.
- **Failure.** Treating a transposed view as contiguous: a kernel that uses `N` as the row stride reads the wrong elements; the shape check passes and the values do not.
- **Bridge.** §11.7 generalizes the swap to every axis.

### 11.7 Permute

- **Connection.** Generalizes §11.6 Transpose from two axes to all axes.
- **Rule.** Reorder all axes according to a permutation $\pi=(\pi_0,\dots,\pi_{d-1})$.
- **Formula.** $\operatorname{shape}(Y)_k=\operatorname{shape}(X)_{\pi_k}$, $\operatorname{stride}(Y)_k=\operatorname{stride}(X)_{\pi_k}$, and $Y_{j_0,\dots,j_{d-1}}=X_{i_0,\dots,i_{d-1}}$ with $i_{\pi_k}=j_k$.
- **Example.** Let $A\in\mathbb R^{2\times3\times4}$ with stride $(12,4,1)$ and $A_{bij}=12b+4i+j$. For $\pi=(2,0,1)$, $P=\operatorname{permute}(A;2,0,1)$ has shape $(4,2,3)$ and stride $(1,12,4)$, and $P_{j,b,i}=A_{b,i,j}$. For example $P_{3,1,2}=A_{1,2,3}=23$, and $3\cdot1+1\cdot12+2\cdot4=23$.
- **Where.** NCHW ↔ NHWC, batch ↔ head ↔ sequence layouts, and any kernel that wants a different axis innermost.
- **Triton.** Permute on the host; in the kernel, reorder the stride terms. `tl.trans` only swaps the last two axes, so a 3-D permute is a sequence of transposes or a 3-D offset expression.
- **Failure.** Confusing $\pi$ with $\pi^{-1}$: the output shape can be right while every element is in the wrong place; test one interior address, not only the shape.
- **Bridge.** §11.8 selects a sub-grid rather than reordering axes.

In [ ]:
S = torch.arange(6).view(2, 1, 3)
Q = S.squeeze(1)
assert S.shape == (2, 1, 3) and S.stride() == (3, 3, 1)
assert Q.shape == (2, 3) and Q.stride() == (3, 1)
assert Q.tolist() == [[0, 1, 2], [3, 4, 5]]
assert torch.equal(Q, S.reshape(2, 3))

v = torch.tensor([10.0, 20.0, 30.0])
assert v[:, None].shape == (3, 1) and v[:, None].stride() == (1, 1)
assert v[None, :].shape == (1, 3) and v[None, :].stride() == (3, 1)
b = v[None, :].expand(2, 3)
assert b.shape == (2, 3) and b.stride() == (0, 1)

X = torch.arange(12).view(3, 4)
Xt = X.t()
assert Xt.shape == (4, 3) and Xt.stride() == (1, 4)
assert Xt.tolist() == [[0, 4, 8], [1, 5, 9], [2, 6, 10], [3, 7, 11]]
assert Xt.data_ptr() == X.data_ptr() and not Xt.is_contiguous()

A = torch.arange(24).view(2, 3, 4)
P = A.permute(2, 0, 1)
assert A.shape == (2, 3, 4) and A.stride() == (12, 4, 1)
assert P.shape == (4, 2, 3) and P.stride() == (1, 12, 4)
assert P[3, 1, 2].item() == A[1, 2, 3].item() == 23
assert 3 * 1 + 1 * 12 + 2 * 4 == 23

### 11.8 Slice

- **Connection.** Builds on the offset+stride model: a slice changes both without moving data.
- **Rule.** Select a regular sub-grid with `start:stop:step` on each axis; `stop` is excluded.
- **Formula.** $Y_{ij}=X_{a+ip,\ b+jq}$ for $0\le i\lt \lceil(c-a)/p\rceil$ and $0\le j\lt \lceil(d-b)/q\rceil$. As a view, $o_Y=o_X+a s_0+b s_1$ and $\operatorname{stride}(Y)=(p s_0,q s_1)$.
- **Example.**

| expression | values | offset | stride |
|---|---|---|---|
| `X[1:3, 1:3]` | `[[5,6],[9,10]]` | $5$ | $(4,1)$ |
| `X[0:3:2, 0:4:2]` | `[[0,2],[8,10]]` | $0$ | $(8,2)$ |

- **Where.** Sliding windows, cropping an image, causal masks, and strided sampling.
- **Triton.** Pass `start`, `step`, `s0`, and `s1` as runtime scalars and compute `src = (start_m + r*step_m)*s0 + ...`; the destination is dense, `dst = r*out_N + c`.
- **Failure.** Off-by-one at `stop` (it is excluded), and mixing source strides with destination strides; PyTorch slicing does not accept a negative step, so use `flip`.
- **Bridge.** §11.9 reverses an axis, a slice with a descending index map.

### 11.9 Reverse indexing

- **Connection.** Builds on §11.8 Slice: reverse is the index map $n-1-i$, not a new primitive.
- **Rule.** Flip one or more axes.
- **Formula.** $Y_{i}=X_{n-1-i}$ on a flipped axis, and $Y_{ij}=X_{M-1-i,\ N-1-j}$ on both. As a view, $o_Y=o_X+(M-1)s_0+(N-1)s_1$ and the strides are $(-s_0,-s_1)$; PyTorch does not expose negative strides, so `flip` returns a copy.
- **Example.**

$$
\operatorname{flip}(X)=
\begin{bmatrix}11&10&9&8\\7&6&5&4\\3&2&1&0\end{bmatrix}.
$$

  $Y_{0,0}=X_{2,3}=11$ and $Y_{2,3}=X_{0,0}=0$.
- **Where.** Reversed convolution kernels, bidirectional sequence models, and reversing a scan order.
- **Triton.** Reverse is a gather with $\pi(i)=n-1-i$: `src = (M-1-r)*s0 + (N-1-c)*s1`; no separate primitive is needed.
- **Failure.** `X[::-1]` raises `ValueError` in PyTorch; reversing the wrong axis, or applying the reverse to the output address instead of the input address, produces a different tensor with the same shape.
- **Bridge.** §11.10 makes the index itself data.

In [ ]:
X = torch.arange(12).view(3, 4)

S = X[1:3, 1:3]
assert S.tolist() == [[5, 6], [9, 10]]
assert S.shape == (2, 2) and S.stride() == (4, 1) and S.storage_offset() == 5
assert not S.is_contiguous()

T = X[0:3:2, 0:4:2]
assert T.tolist() == [[0, 2], [8, 10]]
assert T.shape == (2, 2) and T.stride() == (8, 2)

R = torch.flip(X, dims=(0, 1))
assert R.tolist() == [[11, 10, 9, 8], [7, 6, 5, 4], [3, 2, 1, 0]]
assert R.is_contiguous()
assert R[0, 0].item() == X[2, 3].item() == 11
assert R[2, 3].item() == X[0, 0].item() == 0

try:
    X[::-1]
except ValueError:
    pass
else:
    raise AssertionError("PyTorch slicing must reject a negative step")

### 11.10 Gather

- **Connection.** Builds on Chapter 6 indexing and §11.8 Slice; now the index is a tensor, not a formula.
- **Rule.** Read an element chosen by an index tensor.
- **Formula.** $Y_{ij}=X_{\mathrm{idx}[i,j],\,j}$, and $Y_{k}=X_{\mathrm{idx}[k]}$ in 1-D. The column $j$ is shared: only the row is indexed. `torch.gather` follows the same rule along `dim=0`.
- **Example.**

| `idx` | output row |
|---|---|
| 2 | 30 31 32 |
| 0 | 10 11 12 |
| 3 | 40 41 42 |
| 1 | 20 21 22 |

  With $\mathrm{idx}=\begin{bmatrix}0&1&3\\2&0&1\end{bmatrix}$: $Y_{0,2}=X_{3,2}=42$, $Y_{1,0}=X_{2,0}=30$, and $Y=\begin{bmatrix}10&21&42\\30&11&22\end{bmatrix}$.
- **Where.** Embedding lookup, top-$k$, beam search, and any permutation of rows.
- **Triton.** Load the index as data: `r = tl.load(idx_ptr + k)`, then `tl.load(src_ptr + r*D + col)`; the index tile is an address tile, so it must be an integer.
- **Failure.** An out-of-range index is not checked; it reads a neighboring allocation or faults. Assert `0 <= idx.min()` and `idx.max() < R` on the host, and remember `torch.gather` requires `idx` to have the same rank as `src`.
- **Bridge.** §11.11 inverts the map.

### 11.11 Scatter

- **Connection.** Inverts §11.10 Gather: write each source row to an index-chosen destination.
- **Rule.** Write each source element to an index-chosen destination.
- **Formula.** $\operatorname{dst}[\mathrm{idx}[i,j],\,j]=\operatorname{src}[i,j]$, and the scatter-add adds instead of overwriting: $\operatorname{dst}[\mathrm{idx}[i,j],\,j]\mathrel{+}=\operatorname{src}[i,j]$.
- **Example (unique).** With the $X$ and $\mathrm{idx}$ of §11.10 and a zero destination, $\operatorname{dst}=[[20,21,22],[40,41,42],[10,11,12],[30,31,32]]$.
- **Example (duplicates).**

| source row | `idx` | adds into |
|---|---|---|
| `[1,2]` | 0 | row 0 |
| `[3,4]` | 2 | row 2 |
| `[5,6]` | 0 | row 0 |
| `[7,8]` | 1 | row 1 |
| `[9,10]` | 2 | row 2 |

  The result is $\operatorname{dst}=[[6,8],[7,8],[12,14]]$: row 0 receives source rows 0 and 2, row 1 receives row 3, and row 2 receives rows 1 and 4.
- **Where.** The backward pass of a gather: an embedding gradient scatters token rows into the vocabulary table.
- **Triton.** A plain `tl.store` is correct only when the indices are unique; with duplicates use `tl.atomic_add(out_ptr + r*D + col, v, mask=mask)`. The sum is correct, but the floating-point order is not fixed.
- **Failure.** Using `tl.store` with duplicate indices silently drops all but one update; a scatter-add also needs the destination initialized to zero, and the dtype must support atomics.
- **Bridge.** §11.12 is the whole-slice special case.

### 11.12 Index selection

- **Connection.** Special case of §11.10 Gather: select whole slices along one axis.
- **Rule.** Select whole slices along one axis.
- **Formula.** $Y_{k,:}=X_{\mathrm{idx}[k],:}$, and $Y=\operatorname{index\_select}(X,\ \dim=0,\ \mathrm{idx})$; for $\dim=1$ the indexed axis is the column instead.
- **Example.** For $x=(10,20,30,40)$ and $\mathrm{idx}=(0,2,3)$, $x[\mathrm{idx}]=(10,30,40)$; for the $4\times3$ matrix $X$ above, $X[(0,2),:]$ is rows $0$ and $2$.
- **Where.** Mini-batching, sampling without replacement, and reordering examples by a permutation.
- **Triton.** Use the same row-gather kernel: the index tensor is 1-D and the inner block covers the row width `D`; `index_select` is a gather whose column index is the identity.
- **Failure.** `X[[0,2]]` selects rows and `X[:, [0,2]]` selects columns; passing the number of selected rows as `D` instead of the width reads past each row.
- **Bridge.** §11.13 drops the assumption that the source is contiguous.

In [ ]:
src = torch.tensor([[10, 11, 12], [20, 21, 22], [30, 31, 32], [40, 41, 42]])
idx = torch.tensor([2, 0, 3, 1])
assert src[idx].tolist() == [[30, 31, 32], [10, 11, 12], [40, 41, 42], [20, 21, 22]]

i2 = torch.tensor([[0, 1, 3], [2, 0, 1]])
assert torch.gather(src, 0, i2).tolist() == [[10, 21, 42], [30, 11, 22]]

dst = torch.zeros_like(src)
dst[idx] = src
assert dst.tolist() == [[20, 21, 22], [40, 41, 42], [10, 11, 12], [30, 31, 32]]

s2 = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0], [9.0, 10.0]])
i2dup = torch.tensor([0, 2, 0, 1, 2])
acc = torch.zeros(3, 2)
acc.index_add_(0, i2dup, s2)
assert acc.tolist() == [[6.0, 8.0], [7.0, 8.0], [12.0, 14.0]]

x1 = torch.tensor([10, 20, 30, 40])
sel = torch.tensor([0, 2, 3])
assert x1.index_select(0, sel).tolist() == [10, 30, 40]
assert torch.equal(x1.index_select(0, sel), x1[sel])

### 11.13 Non-contiguous handling

- **Connection.** Collects §11.6–§11.12: every view above can be non-contiguous.
- **Rule.** A kernel must use the tensor's real strides; a contiguous assumption is a silent bug.
- **Formula.** Contiguous means $s_{d-1}=1$, $s_k=n_{k+1}s_{k+1}$, and $o=0$; otherwise the address is the general map $\operatorname{addr}(\mathbf i)=o+\sum_k i_k s_k$.
- **Example.**

| tensor | shape | stride | offset | $\operatorname{addr}(2,1)$ |
|---|---|---|---|---|
| $Y=X^\top$ | $(4,3)$ | $(1,4)$ | $0$ | $2+4=6$ |
| $Z=Y.\operatorname{contiguous}()$ | $(4,3)$ | $(3,1)$ | $0$ | $6+1=7$ |

  $Y_{2,1}=Z_{2,1}=6$, but the two addresses differ. The sliced view $W=X[1{:}3,1{:}3]$ has shape $(2,2)$, strides $(4,1)$, and offset $5$; it is non-contiguous because the row stride is $4$, not $2$.
- **Where.** Avoid hidden copies in a pipeline, and write kernels that accept transposed or sliced views without a `.contiguous()` call.
- **Triton.** Pass `t.stride(0)`, `t.stride(1)`, and the pointer; `t.data_ptr()` already includes the storage offset. Compute `addr = r*s0 + c*s1` and let the mask handle the boundary.
- **Failure.** Assuming `stride(1) == 1` and `stride(0) == N`: the kernel runs and the shape is right, but it reads a transposed or shifted tile. `.view()` fails loudly on a non-contiguous tensor; `.reshape()` hides the copy.
- **Bridge.** The worked example below fixes every number; Parts II–IV turn the maps into kernels.

**Where this leads.**

- The worked example collects all thirteen maps in one place and checks them with `assert`.
- Method 01 writes the gather, scatter-add, and index-select loops in Python as the oracle.
- Method 02 turns them into a tiled transpose, a row gather, and an atomic scatter-add.
- Method 03 fuses the movement with arithmetic and passes explicit strides for non-contiguous views.
- Chapter 12 turns the stride discussion into the layout of a Triton tile.

In [ ]:
X = torch.arange(12).view(3, 4)
Y = X.t()
Z = Y.contiguous()

assert Y.shape == Z.shape == (4, 3)
assert Y.stride() == (1, 4) and Z.stride() == (3, 1)
assert torch.equal(Y, Z)

rows = torch.arange(4)[:, None]
cols = torch.arange(3)[None, :]
assert torch.equal(Y, X[cols, rows])
assert Y[2, 1].item() == X[1, 2].item() == 6
assert 2 * 1 + 1 * 4 == 6 and 2 * 3 + 1 == 7

W = X[1:3, 1:3]
assert W.shape == (2, 2) and W.stride() == (4, 1) and W.storage_offset() == 5
assert not W.is_contiguous()
assert W.reshape(-1).tolist() == [5, 6, 9, 10]

print("non-contiguous: logical strides and contiguous copy verified")

## Worked numerical example

**Thread.** Every number below is checked by the next cell with `assert`; nothing is printed.

- **Step 1 — the spine.** $X=\begin{bmatrix}0&1&2&3\\4&5&6&7\\8&9&10&11\end{bmatrix}$ with stride $(4,1)$ and offset $0$. Because $X=\operatorname{arange}(12)$, the value table and the address table $4i+j$ are identical.

| $i$ \ $j$ | 0 | 1 | 2 | 3 |
|---:|---:|---:|---:|---:|
| **0** | 0 | 1 | 2 | 3 |
| **1** | 4 | 5 | 6 | 7 |
| **2** | 8 | 9 | 10 | 11 |

- **Step 2 — flatten and unflatten.** $\operatorname{flat}(X)=(0,1,2,3,4,5,6,7,8,9,10,11)$, and every pair satisfies $i=f\mathbin{//}4$, $j=f\bmod4$, $f=4i+j$.

| $f$ | 0 | 4 | 7 | 10 | 11 |
|---:|---:|---:|---:|---:|---:|
| $(i,j)$ | $(0,0)$ | $(1,0)$ | $(1,3)$ | $(2,2)$ | $(2,3)$ |

- **Step 3 — reshape, not transpose.** To $(2,6)$: `[[0,1,2,3,4,5],[6,7,8,9,10,11]]`, stride $(6,1)$. To $(4,3)$: `[[0,1,2],[3,4,5],[6,7,8],[9,10,11]]`, stride $(3,1)$. The second is not $X^\top$.
- **Step 4 — transpose.** $X^\top=\begin{bmatrix}0&4&8\\1&5&9\\2&6&10\\3&7&11\end{bmatrix}$ with shape $(4,3)$ and stride $(1,4)$.

| entry | value | logical source | $Y$-address | $X$-address |
|---|---:|---|---:|---:|
| $Y_{0,0}$ | 0 | $X_{0,0}$ | $0$ | $0$ |
| $Y_{1,2}$ | 9 | $X_{2,1}$ | $1+8=9$ | $8+1=9$ |
| $Y_{2,1}$ | 6 | $X_{1,2}$ | $2+4=6$ | $4+2=6$ |
| $Y_{3,2}$ | 11 | $X_{2,3}$ | $3+8=11$ | $8+3=11$ |

- **Step 5 — permute a $2\times3\times4$.** $A_{bij}=12b+4i+j$ with stride $(12,4,1)$; the slices are $A_0=$ `[[0,1,2,3],[4,5,6,7],[8,9,10,11]]` and $A_1=$ `[[12,13,14,15],[16,17,18,19],[20,21,22,23]]`. For $\pi=(2,0,1)$, $P$ has shape $(4,2,3)$ and stride $(1,12,4)$, with $P_{j,b,i}=A_{b,i,j}$.

| $P$ entry | value | source |
|---|---:|---|
| $P_{0,0,0}$ | 0 | $A_{0,0,0}$ |
| $P_{1,0,0}$ | 1 | $A_{0,0,1}$ |
| $P_{0,1,0}$ | 12 | $A_{1,0,0}$ |
| $P_{3,1,2}$ | 23 | $A_{1,2,3}$ |

  The address of $P_{3,1,2}$ is $3\cdot1+1\cdot12+2\cdot4=23$, the same as the address of $A_{1,2,3}$ in the original buffer.
- **Step 6 — slice and reverse.** A slice adds a start to the offset and multiplies the strides by the steps.

| expression | values | offset | stride |
|---|---|---|---|
| `X[1:3, 1:3]` | `[[5,6],[9,10]]` | $5$ | $(4,1)$ |
| `X[0:3:2, 0:4:2]` | `[[0,2],[8,10]]` | $0$ | $(8,2)$ |

  Reversing both axes is the index map $Y_{ij}=X_{2-i,3-j}$:

$$
Y=\begin{bmatrix}11&10&9&8\\7&6&5&4\\3&2&1&0\end{bmatrix},
\qquad Y_{0,0}=X_{2,3}=11.
$$

- **Step 7 — gather.** With $S=$ `[[10,11,12],[20,21,22],[30,31,32],[40,41,42]]` and $\mathrm{idx}=(2,0,3,1)$:

| `idx` | output row |
|---|---|
| 2 | 30 31 32 |
| 0 | 10 11 12 |
| 3 | 40 41 42 |
| 1 | 20 21 22 |

  An elementwise gather uses one index per column; with $\mathrm{idx}=\begin{bmatrix}0&1&3\\2&0&1\end{bmatrix}$, $Y_{0,2}=S_{3,2}=42$, $Y_{1,0}=S_{2,0}=30$, and $Y=[[10,21,42],[30,11,22]]$.
- **Step 8 — scatter and scatter-add.** A scatter with unique indices inverts the gather: $\operatorname{dst}=[[20,21,22],[40,41,42],[10,11,12],[30,31,32]]$. With duplicates:

| source row | `idx` | adds into |
|---|---|---|
| `[1,2]` | 0 | row 0 |
| `[3,4]` | 2 | row 2 |
| `[5,6]` | 0 | row 0 |
| `[7,8]` | 1 | row 1 |
| `[9,10]` | 2 | row 2 |

  $\operatorname{dst}_0=(1+5,\ 2+6)=(6,8)$, $\operatorname{dst}_1=(7,8)$, and $\operatorname{dst}_2=(3+9,\ 4+10)=(12,14)$. Column $j$ never mixes with column $j'$ because the scatter formula indexes only the row.

In [ ]:
X = torch.arange(12).view(3, 4)
assert X.tolist() == [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
assert X.stride() == (4, 1) and X.storage_offset() == 0

flat = X.reshape(-1)
assert flat.tolist() == list(range(12))
assert [i * 4 + j for i in range(3) for j in range(4)] == flat.tolist()
for f, (i, j) in [(0, (0, 0)), (4, (1, 0)), (7, (1, 3)), (10, (2, 2)), (11, (2, 3))]:
    assert f // 4 == i and f % 4 == j

assert X.reshape(2, 6).tolist() == [[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11]]
assert X.reshape(4, 3).tolist() == [[0, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]]
assert X.reshape(4, 3).tolist() != X.t().tolist()

Xt = X.t()
assert Xt.shape == (4, 3) and Xt.stride() == (1, 4)
assert Xt.tolist() == [[0, 4, 8], [1, 5, 9], [2, 6, 10], [3, 7, 11]]
assert Xt[1, 2].item() == X[2, 1].item() == 9
assert Xt[2, 1].item() == X[1, 2].item() == 6
assert 2 * 1 + 1 * 4 == 1 * 4 + 2 == 6

A = torch.arange(24).view(2, 3, 4)
P = A.permute(2, 0, 1)
assert A.shape == (2, 3, 4) and A.stride() == (12, 4, 1)
assert P.shape == (4, 2, 3) and P.stride() == (1, 12, 4)
assert P[0, 0, 0].item() == A[0, 0, 0].item() == 0
assert P[1, 0, 0].item() == A[0, 0, 1].item() == 1
assert P[0, 1, 0].item() == A[1, 0, 0].item() == 12
assert P[3, 1, 2].item() == A[1, 2, 3].item() == 23
assert 3 * 1 + 1 * 12 + 2 * 4 == 23

S = X[1:3, 1:3]
assert S.tolist() == [[5, 6], [9, 10]]
assert S.shape == (2, 2) and S.stride() == (4, 1) and S.storage_offset() == 5
T = X[0:3:2, 0:4:2]
assert T.tolist() == [[0, 2], [8, 10]] and T.stride() == (8, 2)
R = torch.flip(X, dims=(0, 1))
assert R.tolist() == [[11, 10, 9, 8], [7, 6, 5, 4], [3, 2, 1, 0]]
assert R[0, 0].item() == X[2, 3].item() == 11
assert R[2, 3].item() == X[0, 0].item() == 0

src = torch.tensor([[10, 11, 12], [20, 21, 22], [30, 31, 32], [40, 41, 42]])
idx = torch.tensor([2, 0, 3, 1])
assert src[idx].tolist() == [[30, 31, 32], [10, 11, 12], [40, 41, 42], [20, 21, 22]]
i2 = torch.tensor([[0, 1, 3], [2, 0, 1]])
assert torch.gather(src, 0, i2).tolist() == [[10, 21, 42], [30, 11, 22]]

dst = torch.zeros_like(src)
dst[idx] = src
assert dst.tolist() == [[20, 21, 22], [40, 41, 42], [10, 11, 12], [30, 31, 32]]
s2 = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0], [9.0, 10.0]])
i2dup = torch.tensor([0, 2, 0, 1, 2])
acc = torch.zeros(3, 2)
acc.index_add_(0, i2dup, s2)
assert acc.tolist() == [[6.0, 8.0], [7.0, 8.0], [12.0, 14.0]]

assert src.index_select(0, torch.tensor([0, 2])).tolist() == [[10, 11, 12], [30, 31, 32]]

Z = Xt.contiguous()
assert Z.shape == (4, 3) and Z.stride() == (3, 1)
assert torch.equal(Xt, Z)
assert 2 * 1 + 1 * 4 == 6 and 2 * 3 + 1 == 7
assert not S.is_contiguous()

## Part II — Method 01: the host reference

**Thread.** One Python iteration per row is the oracle for the kernels.

- **What it is.** A deliberately scalar implementation: one Python iteration per selected row or element, using ordinary indexing.
- **Why scalar.** It is the correctness oracle for Method 02; unlike the kernels, it may allocate, may use `int64` indices, and may add in a fixed order.
- **Functions.**
  - `ref_gather_rows` — read one row per index (a gather).
  - `ref_scatter_add` — add one row per index (a scatter-add).
  - `ref_index_select` — the whole-slice special case.
- **CPU check.** The next cell checks each function against PyTorch on CPU, which is why the chapter is runnable without an accelerator.
- **Bridge.** Part III turns each loop into a Triton kernel.

In [ ]:
def ref_gather_rows(src, idx):
    # Method 01: one Python iteration per output row.
    out = torch.empty((idx.numel(), src.shape[1]), dtype=src.dtype)
    for k in range(idx.numel()):
        out[k] = src[int(idx[k])]
    return out


def ref_scatter_add(n_rows, src, idx):
    # Method 01: add one source row at a time; duplicates accumulate.
    out = torch.zeros((n_rows, src.shape[1]), dtype=src.dtype)
    for k in range(src.shape[0]):
        out[int(idx[k])] += src[k]
    return out


def ref_index_select(x, idx):
    # Method 01: the whole-slice special case (dim=0).
    out = torch.empty((idx.numel(),) + tuple(x.shape[1:]), dtype=x.dtype)
    for k in range(idx.numel()):
        out[k] = x[int(idx[k])]
    return out


src = torch.tensor([[10, 11, 12], [20, 21, 22], [30, 31, 32], [40, 41, 42]])
idx = torch.tensor([2, 0, 3, 1])
assert torch.equal(ref_gather_rows(src, idx), src[idx])
assert torch.equal(ref_index_select(src, idx), src.index_select(0, idx))
assert torch.equal(ref_gather_rows(src, torch.tensor([1, 1, 0])),
                   src[torch.tensor([1, 1, 0])])

s2 = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0], [9.0, 10.0]])
i2 = torch.tensor([0, 2, 0, 1, 2])
ref = torch.zeros(3, 2)
ref.index_add_(0, i2, s2)
assert torch.equal(ref_scatter_add(3, s2, i2), ref)

# Elementwise row gather along dim=0, one element at a time.
i3 = torch.tensor([[0, 1, 3], [2, 0, 1]])
naive = torch.empty((2, 3))
for r in range(2):
    for c in range(3):
        naive[r, c] = src[int(i3[r, c]), c]
assert torch.equal(naive, torch.gather(src, 0, i3))

print("method 01: gather, scatter-add, and index_select match PyTorch on CPU")

## Part III — Method 02: Triton kernels

**Thread.** Each kernel maps a program to a tile or a row, masks the tail, and loads an index tile as data.

| kernel | grid | work per program | reference |
|---|---|---|---|
| transpose | $(\lceil M/B_M\rceil,\lceil N/B_N\rceil)$ | load a $B_M\times B_N$ tile, `tl.trans`, store | `x.t()` |
| row gather | $(R,)$ | one output row: load `idx[k]`, then `D` values | `src[idx]` |
| scatter-add | $(S,)$ | one source row: atomic-add into row `idx[k]` | `index_add_` |

- **Masks.** The transpose masks both the load tile and the transposed store tile; the gather and scatter mask the column tail with `cols < D`. The masked load uses `other=0.0` because zero is the additive identity for the scatter-add and a safe filler for the gather output.
- **Why atomics.** A gather may repeat an index, which is harmless; a plain scatter with repeated indices is a race where the last writer wins. `tl.atomic_add` makes the update read-modify-write, so every contribution survives.
- **Bridge.** Part IV fuses the same movements with arithmetic.

In [ ]:
@triton.jit
def m11_transpose_kernel(x_ptr, out_ptr, M, N,
                         BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)

    mask = (rows[:, None] < M) & (cols[None, :] < N)
    tile = tl.load(x_ptr + rows[:, None] * N + cols[None, :],
                   mask=mask, other=0.0)
    tile_t = tl.trans(tile)

    mask_t = (cols[:, None] < N) & (rows[None, :] < M)
    tl.store(out_ptr + cols[:, None] * M + rows[None, :], tile_t, mask=mask_t)


def transpose(x, BLOCK_M=16, BLOCK_N=16):
    M, N = x.shape
    out = torch.empty((N, M), device=DEVICE, dtype=x.dtype)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    m11_transpose_kernel[grid](x, out, M, N,
                               BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out


def test_transpose():
    require_gpu("transpose")
    x = torch.arange(17 * 29, device=DEVICE, dtype=torch.float32).view(17, 29)
    check("transpose tile", transpose(x), x.t())
    xs = torch.arange(12, device=DEVICE, dtype=torch.float32).view(3, 4)
    check("transpose spine", transpose(xs, BLOCK_M=4, BLOCK_N=4), xs.t())


run_if_gpu(test_transpose)

In [ ]:
@triton.jit
def m11_gather_rows_kernel(src_ptr, idx_ptr, out_ptr, D,
                           BLOCK_D: tl.constexpr):
    k = tl.program_id(0)
    cols = tl.arange(0, BLOCK_D)
    mask = cols < D
    src_row = tl.load(idx_ptr + k)
    value = tl.load(src_ptr + src_row * D + cols, mask=mask, other=0.0)
    tl.store(out_ptr + k * D + cols, value, mask=mask)


def gather_rows(src, idx, BLOCK_D=None):
    D = src.shape[1]
    block = BLOCK_D if BLOCK_D is not None else triton.next_power_of_2(D)
    out = torch.empty((idx.numel(), D), device=src.device, dtype=src.dtype)
    m11_gather_rows_kernel[(idx.numel(),)](src, idx, out, D, BLOCK_D=block)
    return out


def test_gather_rows():
    require_gpu("gather rows")
    src = torch.randn(6, 13, device=DEVICE)
    idx = torch.tensor([4, 1, 1, 0, 5, 3], device=DEVICE, dtype=torch.int32)
    check("gather rows", gather_rows(src, idx), src[idx.long()])


run_if_gpu(test_gather_rows)

In [ ]:
@triton.jit
def m11_scatter_add_kernel(src_ptr, idx_ptr, out_ptr, D,
                           BLOCK_D: tl.constexpr):
    k = tl.program_id(0)
    cols = tl.arange(0, BLOCK_D)
    mask = cols < D
    dst_row = tl.load(idx_ptr + k)
    value = tl.load(src_ptr + k * D + cols, mask=mask, other=0.0)
    tl.atomic_add(out_ptr + dst_row * D + cols, value, mask=mask)


def scatter_add(n_rows, src, idx, BLOCK_D=None):
    D = src.shape[1]
    block = BLOCK_D if BLOCK_D is not None else triton.next_power_of_2(D)
    out = torch.zeros((n_rows, D), device=src.device, dtype=src.dtype)
    m11_scatter_add_kernel[(src.shape[0],)](src, idx, out, D, BLOCK_D=block)
    return out


def test_scatter_add():
    require_gpu("scatter add")
    src = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0],
                        [7.0, 8.0], [9.0, 10.0]], device=DEVICE)
    idx = torch.tensor([0, 2, 0, 1, 2], device=DEVICE, dtype=torch.int32)
    ref = torch.zeros(3, 2, device=DEVICE)
    ref.index_add_(0, idx.long(), src)
    check("scatter-add duplicates", scatter_add(3, src, idx), ref)
    print("scatter-add: duplicate rows are summed by atomics")


run_if_gpu(test_scatter_add)

## Part IV — Method 03: fused transform plus compute

**Thread.** Move the data once and do the arithmetic in registers.

- **Gather plus affine.** Read one row per index and apply $y=\alpha x+\beta$ before the store.
- **Strided transpose plus activation.** Read a non-contiguous view with its real strides and apply $\max(x,0)$ on the way out.
- **Traffic.** For $M$ output rows of width $D$ and $b$ bytes per element, unfused gather-then-scale moves about $Q_{\text{unfused}}\approx4MDb$; the fused kernel moves about $Q_{\text{fused}}\approx2MDb$. The arithmetic is free by comparison; the second pass is not.
- **Why explicit strides.** The fused kernel does not care whether the input is contiguous: it receives $(s_0,s_1)$ and the pointer to the view's first element, then computes the address with the general map. The host must pass the *view's* strides, not the base tensor's.
- **Bridge.** The adjoint section turns the forward maps into backward passes.

In [ ]:
@triton.jit
def m11_gather_affine_kernel(src_ptr, idx_ptr, out_ptr, D, scale, bias,
                             BLOCK_D: tl.constexpr):
    k = tl.program_id(0)
    cols = tl.arange(0, BLOCK_D)
    mask = cols < D
    src_row = tl.load(idx_ptr + k)
    value = tl.load(src_ptr + src_row * D + cols, mask=mask, other=0.0)
    tl.store(out_ptr + k * D + cols, value * scale + bias, mask=mask)


def gather_affine(src, idx, scale, bias, BLOCK_D=None):
    D = src.shape[1]
    block = BLOCK_D if BLOCK_D is not None else triton.next_power_of_2(D)
    out = torch.empty((idx.numel(), D), device=src.device, dtype=src.dtype)
    m11_gather_affine_kernel[(idx.numel(),)](src, idx, out, D, scale, bias,
                                              BLOCK_D=block)
    return out


def test_gather_affine():
    require_gpu("fused gather+affine")
    src = torch.arange(35, device=DEVICE, dtype=torch.float32).view(5, 7)
    idx = torch.tensor([4, 1, 1, 0, 3], device=DEVICE, dtype=torch.int32)
    ref = src[idx.long()] * 2.5 - 1.0
    check("fused gather+affine", gather_affine(src, idx, 2.5, -1.0), ref)


run_if_gpu(test_gather_affine)

In [ ]:
@triton.jit
def m11_strided_relu_kernel(x_ptr, out_ptr, M, N, s0, s1,
                            BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    flat = pid * BLOCK + tl.arange(0, BLOCK)
    mask = flat < M * N
    row = flat // N
    col = flat % N
    value = tl.load(x_ptr + row * s0 + col * s1, mask=mask, other=0.0)
    tl.store(out_ptr + flat, tl.maximum(value, 0.0), mask=mask)


def strided_relu(x, BLOCK=64):
    M, N = x.shape
    out = torch.empty(M * N, device=x.device, dtype=x.dtype)
    grid = (triton.cdiv(M * N, BLOCK),)
    m11_strided_relu_kernel[grid](x, out, M, N,
                                  x.stride(0), x.stride(1), BLOCK=BLOCK)
    return out


def test_strided_relu():
    require_gpu("strided relu")
    base = torch.randn(9, 13, device=DEVICE)
    view = base.t()
    check("strided transpose+relu", strided_relu(view).view(13, 9),
          view.contiguous().relu())

    X = torch.randn(7, 11, device=DEVICE)
    W = X[1:6, 2:9]
    check("strided slice+relu", strided_relu(W).view(5, 7),
          W.contiguous().relu())
    print("strided: explicit (s0, s1) handles transposed and sliced views")


run_if_gpu(test_strided_relu)

### Adjoints of the transforms

**Thread.** Every transform here is linear in the values, so its adjoint is the transpose of the linear map.

- **General rule.** If $Y=PX$, then $\frac{\partial L}{\partial X}=P^\top\frac{\partial L}{\partial Y}$.
- **Transpose.** Its own adjoint: $Y=X^\top\Rightarrow dX=dY^\top$.
- **Permute.** Inverted by the inverse permutation.
- **Slice.** Inverted by zero-padding the gradient back into the full shape.
- **Reverse.** Its own adjoint: flipping the gradient flips it back.
- **Gather.** Inverted by **scatter-add**: $Y_k=X_{\mathrm{idx}[k]}\Rightarrow dX_{\mathrm{idx}[k]}\mathrel{+}=dY_k$.
- **Why it matters.** The last line is the whole reason the embedding backward is a scatter-add. The next cell checks the transpose, gather, and reverse adjoints against autograd.

In [ ]:
base = torch.arange(12.0, requires_grad=True)
x = base.view(3, 4)
y = x.t()
upstream = torch.arange(12.0, 0, -1).view(4, 3)
y.backward(upstream)
assert torch.equal(base.grad.view(3, 4), upstream.t())

v = torch.arange(5.0, requires_grad=True)
idx = torch.tensor([0, 2, 2, 4])
out = v[idx]
out.backward(torch.tensor([1.0, 10.0, 100.0, 1000.0]))
assert v.grad.tolist() == [1.0, 0.0, 110.0, 0.0, 1000.0]

w = torch.arange(6.0, requires_grad=True)
flipped = torch.flip(w, dims=(0,))
flipped.backward(torch.arange(6.0))
assert w.grad.tolist() == [5.0, 4.0, 3.0, 2.0, 1.0, 0.0]

print("adjoints: transpose, gather (scatter-add), and reverse match autograd")

## Pitfalls

**Thread.** Each row is a shape/stride bug that can pass the shape check while the values are wrong.

| Pitfall | What happens | Fix |
|---|---|---|
| `reshape` versus `view` | `view` raises on a non-contiguous tensor; `reshape` copies silently | choose deliberately and assert `shape`/`stride` |
| Transposed view treated as contiguous | the kernel uses `N` instead of `stride(0)`, so values are permuted but shapes match | pass `t.stride(0)`, `t.stride(1)` |
| Gather index out of bounds | undefined read or a fault; no Python exception | assert `0 <= idx.min()` and `idx.max() < R` |
| Scatter collisions | a plain `tl.store` keeps only the last writer | use `tl.atomic_add` or a sorted segmented reduction |
| Negative indices | PyTorch wraps `-1`; Triton pointer arithmetic does not | normalize with `idx % R` or reject negatives |
| Flatten order | the flat order is logical row-major, not storage order | flatten with `reshape`, never with `view`, on a view |
| Permute axes | `dims` is the new axis order, not its inverse | test one interior address, not only the shape |
| Squeeze the wrong axis | a valid shape with the wrong layout | assert both `shape` and `stride` |
| Slice endpoint | `stop` is excluded; `step < 0` raises | check the count $\lceil(c-a)/p\rceil$ |
| Index dtype | a float index is truncated or rejected | cast to `torch.int32` or `torch.int64` before launch |

## Self-check

**Thread.** Answer each from the index map, not from memory.

1. For the spine, write the address of $X_{2,1}$ and of $X^\top_{1,2}$. Are they the same? Which tensor has stride $(1,4)$?
2. Flatten the spine with $f=iN+j$. What are $f(2,3)$, $f(1,2)$, and the coordinates of $f=9$?
3. Reshape the spine to $(4,3)$ and transpose it. Why are the two results different even though both have shape $(4,3)$?
4. A tensor has shape $(2,1,3)$ and strides $(3,3,1)$. Give the shape and stride after `squeeze(1)` and after `squeeze(0)`. Which call is an error?
5. For $A\in\mathbb R^{2\times3\times4}$ with stride $(12,4,1)$, give the shape and stride of `A.permute(2, 0, 1)`. Where is $A_{1,2,3}$ in the result?
6. `X[1:3, 1:3]` has shape $(2,2)$. Why is its stride $(4,1)$ and not $(2,1)$? What is its storage offset?
7. Write the gather formula and the scatter-add formula. Why is the backward of a gather a scatter-add, and not another gather?
8. A kernel assumes `stride(1) == 1`. Name two views for which that is false and explain what the kernel reads instead.

## Summary

**Thread.** Every row is the same index-map question with a different answer.

- A tensor is $(shape, stride, offset)$; a view changes those three numbers, and a copy allocates a contiguous buffer.
- Flatten and unflatten are the maps $f=iN+j$ and $(i,j)=(f\mathbin{//}N,f\bmod N)$.
- Reshape preserves the flat order; transpose and permute change the index map. A reshape is not a transpose.
- Squeeze removes a size-1 axis and leaves the other strides alone; expand adds an axis, and `expand` sets its stride to zero.
- Slice adds a start to the offset and scales the strides by the steps. Reverse is the index map $n-1-i$; PyTorch's `flip` copies.
- Gather is $Y_{ij}=X_{\mathrm{idx}[i,j],j}$; scatter is its inverse; scatter-add is its adjoint. Duplicate indices require atomics.
- A Triton kernel never infers a layout: pass the real strides, or pass a contiguous buffer on purpose.
- Fusion removes a full read/write pass; it does not change the index map.

**Where this leads.**

- The exercises apply the same maps to column-major flatten, inverse permutations, strided slices, bounds, deterministic scatter-add, and fusion traffic.
- Chapter 12 turns the stride discussion into the layout of a Triton tile.
- The embedding chapter applies the gather/scatter-add pair to a vocabulary table.

**Next:** [Chapter 12 — Triton Layout Fundamentals](../../part5_layouts/ch12_layouts/12_complete.ipynb)

## Exercises

**Thread.** Each exercise changes one index map and checks it; none needs a new kernel primitive.

1. **Flatten order.** Write a function that flattens a $3\times4$ matrix in column-major order and verify it against `X.t().reshape(-1)`.
2. **Unflatten by hand.** For $M=4$, $N=5$, list $(i,j)$ for $f=0,6,12,19$ and check with integer division.
3. **Reshape or transpose.** For the spine, print (as an assert) the difference between `X.reshape(4, 3)` and `X.t()` and state which one preserves the flat order.
4. **Squeeze audit.** Take a tensor of shape $(2,1,3,1)$, squeeze both singleton axes, and assert the resulting shape and stride.
5. **Permute inverse.** For $\pi=(2,0,1)$, find $\pi^{-1}$ and verify that `A.permute(*pi).permute(*pi_inv)` is `A`.
6. **Strided slice.** Copy `X[0:3:2, 1:4:2]` with a Triton kernel that takes `start_m, step_m, start_n, step_n` and compare with PyTorch.
7. **Reverse one axis.** Write a kernel for `torch.flip(x, dims=(1,))` and check that reversing twice is the identity.
8. **Gather bounds.** Build an index tensor with one out-of-range value, show that the host check catches it, and add a masked variant that drops the row.
9. **Deterministic scatter-add.** Replace the atomics with a sort by destination index followed by a segmented sum, and compare the result bitwise across runs.
10. **Fusion traffic.** For $M=1024$, $D=768$, and `float32`, compute the bytes moved by gather-then-scale and by the fused kernel. How much traffic does fusion remove?

Hints are in [`../../docs/SOLUTIONS.md`](../../docs/SOLUTIONS.md).

In [ ]:
def test_cpu_suite():
    X = torch.arange(12).view(3, 4)
    assert X.reshape(-1).tolist() == list(range(12))
    assert X.t().stride() == (1, 4)
    idx = torch.tensor([2, 0, 3, 1])
    src = torch.arange(16.0).view(4, 4)
    assert torch.equal(src[idx], torch.stack([src[2], src[0], src[3], src[1]]))
    dst = torch.zeros_like(src)
    dst.index_add_(0, torch.tensor([0, 2, 0, 1]), src[:4])
    assert dst[0].tolist() == (src[0] + src[2]).tolist()


def test_gpu_suite():
    test_transpose()
    test_gather_rows()
    test_scatter_add()
    test_gather_affine()
    test_strided_relu()


test_cpu_suite()
print("CPU: shape/stride and index-map checks passed")
run_if_gpu(test_gpu_suite)